## Enrollment Forecasting Notebook(Proof of Concept)

**Notes**
 * In data cleaning section, the Not Applicable values were replaced with 0s. This is because Not Applicable is the same as having 0 enrolled, other than for contextual reasons for the reader.
 * The data will be compiled into one dataframe, which will be used to forecast enrollment for each state.

**Data Cleaning**

* Columns in final DataFrame are as follows:
  * 'State or jurisdiction' : The state or jurisdiction of the row
  * 'Total_Pub_Under' : Total public undergrad enrollment
  * '4y_Pub_Under' : 4-year public undergrad enrollment
  * '2y_Pub_Under' : 2-year public undergrad enrollment
  * 'Total_Pub_Postbacc' : Total public postbacc enrollment
  * 'Total_Priv_Under' : Total private undergrad enrollment
  * 'Np_4y_Priv_Under' : Nonprofit 4-year private undergrad enrollment
  * 'Fp_4y_Priv_Under' : For-profit 4-year private undergrad enrollment
  * 'Np_2y_Priv_Under' : Nonprofit 2-year private undergrad enrollment
  * 'Fp_2y_Priv_Under' : For-profit 2-year private undergrad enrollment
  * 'Total_Priv_Postbacc' : Total private postbacc enrollment
  * 'Np_4y_Priv_Postbacc' : Nonprofit 4-year private postbacc enrollment
  * 'Fp_4y_Priv_Postbacc' : For-profit 4-year private postbacc enrollment


**EDA**

* Utilized ydata_profiling for exploratory analysis
* Obviously highly correlated features, especially totals, as they are combinations of other columns. Will hopefully be adding years, so that we can analyze trends over time.
* Full enrollment data including year as well as data shown here will be created in the data cleaning ipynb, as well as the eda ipynb.

## Data Cleaning

In [18]:
import numpy as np
import pandas as pd
from functools import reduce

In [19]:
data = pd.read_excel('data/enroll2021-22.xlsx', skiprows=1)
data.head()

,State or jurisdiction,Public,Public.1,Public.2,Public.3,Private,Private.1,Private.2,Private.3,Private.4,Private.5,Private.6,Private.7
0,State or jurisdiction,Undergraduate,Undergraduate,Undergraduate,Postbaccalaureate,Undergraduate,Undergraduate,Undergraduate,Undergraduate,Undergraduate,Postbaccalaureate,Postbaccalaureate,Postbaccalaureate
1,State or jurisdiction,Total,4-year,2-year,Postbaccalaureate,Total,Nonprofit 4-year,For-profit 4-year,Nonprofit 2-year,For-profit 2-year,Total,Nonprofit 4-year,For-profit 4-year
2,1,2,3,4,5,6,7,8,9,10,11,12,13
3,United States,11944633,7466861,4477772,1598891,3503787,2698780,600183,27574,177250,1612540,1386750,225790
4,Alabama,207241,131801,75440,42939,32151,18116,13201,295,539,10429,5478,4951


In [20]:
public_undergrad = data[['State or jurisdiction', 'Public', 'Public.1', 'Public.2']]
public_undergrad = public_undergrad.drop([0,1,2])
public_undergrad.columns = ['State or jurisdiction', 'Total', '4-year', '2-year']
public_undergrad.dropna(inplace=True)
public_undergrad

,State or jurisdiction,Total,4-year,2-year
3,United States,11944633,7466861,4477772
4,Alabama,207241,131801,75440
5,Alaska,18128,18128,†
6,Arizona,322815,170336,152479
7,Arkansas,112180,72905,39275
...,...,...,...,...
60,Marshall Islands,1267,1267,†
61,Northern Marianas,1298,1298,†
62,Palau,525,†,525
63,Puerto Rico,43351,42409,942


In [21]:
num = public_undergrad['4-year'][3] + public_undergrad['2-year'][3]
print(num)

11944633


In [22]:
public_pb = data[['State or jurisdiction', 'Public.3']]
public_pb = public_pb.drop([0,1,2])
public_pb.columns = ['State or jurisdiction', 'Total_PB']
public_pb.dropna(inplace=True)
public_pb

,State or jurisdiction,Total_PB
3,United States,1598891
4,Alabama,42939
5,Alaska,1808
6,Arizona,43693
7,Arkansas,18970
...,...,...
60,Marshall Islands,†
61,Northern Marianas,†
62,Palau,†
63,Puerto Rico,5748


In [23]:
private_undergrad = data[['State or jurisdiction', 'Private'] + [name for name in data.columns if name.startswith('Private') and name.endswith(('1', '2', '3', '4'))]]
private_undergrad = private_undergrad.drop([0,1,2])
private_undergrad.columns = ['State or jurisdiction', 'Total', 'Nonprofit 4-year',	'For-profit 4-year','Nonprofit 2-year',	'For-profit 2-year']
private_undergrad.dropna(inplace=True)
private_undergrad

,State or jurisdiction,Total,Nonprofit 4-year,For-profit 4-year,Nonprofit 2-year,For-profit 2-year
3,United States,3503787,2698780,600183,27574,177250
4,Alabama,32151,18116,13201,295,539
5,Alaska,815,420,†,78,317
6,Arizona,179110,5802,159804,†,13504
7,Arkansas,13785,12438,355,977,15
...,...,...,...,...,...,...
60,Marshall Islands,†,†,†,†,†
61,Northern Marianas,†,†,†,†,†
62,Palau,†,†,†,†,†
63,Puerto Rico,98665,67328,27089,†,4248


In [24]:
private_pb = data[['State or jurisdiction'] + [name for name in data.columns if name.startswith('Private') and name.endswith(('5', '6', '7'))]]
private_pb = private_pb.drop([0,1,2])
private_pb.columns = ['State or jurisdiction', 'Total', 'Nonprofit 4-year',	'For-profit 4-year']
private_pb.dropna(inplace=True)
private_pb

,State or jurisdiction,Total,Nonprofit 4-year,For-profit 4-year
3,United States,1612540,1386750,225790
4,Alabama,10429,5478,4951
5,Alaska,178,178,†
6,Arizona,68528,6154,62374
7,Arkansas,2821,2774,47
...,...,...,...,...
60,Marshall Islands,†,†,†
61,Northern Marianas,†,†,†
62,Palau,†,†,†
63,Puerto Rico,20536,17419,3117


In [25]:
for df in [public_pb, private_pb, public_undergrad, private_undergrad]:
    df.replace('†', 0, inplace=True)
    df = df.reset_index(drop=True, inplace=True)

/var/folders/6g/_wpr8bs53rs84lqb6nn_rk900000gn/T/ipykernel_49679/1509617434.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace('†', 0, inplace=True)


In [26]:
public_undergrad

,State or jurisdiction,Total,4-year,2-year
0,United States,11944633,7466861,4477772
1,Alabama,207241,131801,75440
2,Alaska,18128,18128,0
3,Arizona,322815,170336,152479
4,Arkansas,112180,72905,39275
...,...,...,...,...
57,Marshall Islands,1267,1267,0
58,Northern Marianas,1298,1298,0
59,Palau,525,0,525
60,Puerto Rico,43351,42409,942


In [27]:
dfs = [public_undergrad, public_pb, private_undergrad, private_pb]

merged_df = reduce(
    lambda left, right: pd.merge(left, right, on='State or jurisdiction', how='outer'),
    dfs
)

In [28]:
merged_df.columns = [
    'State or jurisdiction',
    'Total_Pub_Under', 
    '4y_Pub_Under', 
    '2y_Pub_Under', ''
    'Total_Pub_Postbacc',
    'Total_Priv_Under',
    'Np_4y_Priv_Under',
    'Fp_4y_Priv_Under',
    'Np_2y_Priv_Under',
    'Fp_2y_Priv_Under',
    'Total_Priv_Postbacc',
    'Np_4y_Priv_Postbacc',
    'Fp_4y_Priv_Postbacc'
    ]
merged_df

,State or jurisdiction,Total_Pub_Under,4y_Pub_Under,2y_Pub_Under,Total_Pub_Postbacc,Total_Priv_Under,Np_4y_Priv_Under,Fp_4y_Priv_Under,Np_2y_Priv_Under,Fp_2y_Priv_Under,Total_Priv_Postbacc,Np_4y_Priv_Postbacc,Fp_4y_Priv_Postbacc
0,Alabama,207241,131801,75440,42939,32151,18116,13201,295,539,10429,5478,4951
1,Alaska,18128,18128,0,1808,815,420,0,78,317,178,178,0
2,American Samoa,1301,1301,0,0,0,0,0,0,0,0,0,0
3,Arizona,322815,170336,152479,43693,179110,5802,159804,0,13504,68528,6154,62374
4,Arkansas,112180,72905,39275,18970,13785,12438,355,977,15,2821,2774,47
...,...,...,...,...,...,...,...,...,...,...,...,...,...
57,West Virginia,60591,48139,12452,11250,49731,5973,40642,0,3116,10708,1154,9554
58,Wisconsin,234041,149295,84746,25779,40576,38671,1733,0,172,15092,14996,96
59,Wyoming,27095,18402,8693,2610,742,0,0,0,742,0,0,0
60,Other jurisdictions,55498,54031,1467,6336,98747,67410,27089,0,4248,20536,17419,3117


## EDA

In [29]:
!pip install ydata_profiling
#Useful for EDA requires python version 3.12 tho

In [30]:
import sys
print(sys.executable)

/Users/arocha/miniconda3/envs/enrollenv/bin/python


In [31]:
!{sys.executable} -m pip install ydata-profiling

In [32]:
from ydata_profiling import ProfileReport

In [33]:
profile = ProfileReport(
    merged_df,
    title="State Enrollment Data Analysis",
    html={'style':{'full_width':True}},
    minimal=False)
profile.to_notebook_iframe()

Render HTML: 100%|██████████| 1/1 [00:00<00:00,  4.55it/s]
